[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/42_gradient_clipping_solution.ipynb)

# 🟡 Solution: Global-Norm Gradient Clipping

*Training · Medium*

Reference implementation. Try it yourself in `42_gradient_clipping.ipynb` first.

---
Clip a gradient pytree by its **global** norm and return the rescaled pytree.

$$\|g\|_2 = \sqrt{\sum_{p \in \text{leaves}} \sum_i g_{p,i}^2}
\qquad
\hat g = g \cdot \min\!\left(1, \frac{\tau}{\|g\|_2}\right)$$

One scalar $\|g\|_2$ is computed across **all** parameters concatenated into a
single vector, and one scalar factor is applied to **every** leaf.

### Rules
- Signature: `clip_by_global_norm(grads, max_norm)`
- `grads` is an arbitrary pytree (nested dicts/lists/tuples, or an `nnx.State`)
- Return the tuple `(clipped_grads, global_norm)` where `global_norm` is the
  norm **before** clipping, as a JAX scalar
- The returned tree must have the **same structure** as the input
- Banned: `optax.clip_by_global_norm`, `optax.global_norm`
- No Python `if` on the norm — the function must survive `jax.jit`
- All-zero gradients must return all-zero gradients, **not** `NaN`

### Why global, and not per-tensor
Per-tensor clipping (`each leaf independently scaled to at most tau`) uses a
*different* multiplier per leaf. That does not shrink the update — it
**rotates** it. In the flattened parameter space the direction of the update is
a unit vector; multiplying block $A$ by 0.1 and block $B$ by 1.0 produces a
descent direction that is no longer parallel to $-\nabla L$, so you are no
longer doing gradient descent on $L$ at all. You are doing gradient descent on
some silently reweighted objective whose per-layer weights change every step.

Global-norm clipping is the exact **Euclidean projection** of $g$ onto the ball
$\{v : \|v\|_2 \le \tau\}$: it is the closest point in the ball, it is a
*positive* multiple of $g$, and therefore
$\cos(\hat g, g) = 1$ exactly. Only the step *length* changes. That is the whole
point — a loss spike should shorten your step, not steer it somewhere else.

Two more consequences worth being able to say out loud:

- The threshold $\tau$ is a property of the **model**, not of a tensor. Under
  per-tensor clipping a good $\tau$ depends on each tensor's fan-in and element
  count, so it silently changes meaning when you widen a layer.
- Clipping is **not** a no-op under Adam. Adam normalises by a *running*
  second moment, so a single clipped step still lowers that step's contribution
  to $\hat v$ and damps the spike for many steps afterwards.

### The two traps
`if global_norm > max_norm: ...` raises `TracerBoolConversionError` the moment
you `jit` it — the norm is a traced value with no concrete truth value. Use
`jnp.minimum` (or `jnp.where`), which is branch-free and compiles to a select.

And every gradient can be exactly zero — a fully masked batch, a frozen
submodule, a dead ReLU block. Then the direct
`g * max_norm / global_norm` evaluates $0 \cdot \infty = \text{NaN}$ and poisons
every parameter on the next update. Writing the factor as
`jnp.minimum(1.0, max_norm / (global_norm + 1e-6))` fixes it twice over: the
`minimum` selects the safe branch, and the epsilon keeps the quotient finite in
the first place.

That epsilon earns its keep in one more place. $\sqrt{\cdot}$ has an *infinite*
derivative at zero, so `jax.grad` of `jnp.sqrt(jnp.sum(g ** 2))` at `g = 0`
returns `NaN` — an unguarded norm is a landmine the moment clipping sits inside
anything you differentiate through, which is exactly where it ends up in
meta-learning and in differentiable-optimizer research code.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def clip_by_global_norm(grads, max_norm):
    leaves = jax.tree.leaves(grads)

    # ONE norm for the whole tree: treat every parameter as one long vector.
    sq_sum = sum(jnp.sum(jnp.square(leaf)) for leaf in leaves)
    global_norm = jnp.sqrt(sq_sum)

    # Branch-free so it survives jit; the eps stops 0/0 -> NaN on zero grads.
    # min(1, tau/||g||) leaves the tree untouched when it is already inside
    # the ball, and is exactly 1.0 in that case so `g * scale is g` numerically.
    scale = jnp.minimum(1.0, max_norm / (global_norm + 1e-6))

    # The SAME scalar hits every leaf -> the direction is preserved exactly.
    clipped = jax.tree.map(lambda g: g * scale, grads)
    return clipped, global_norm

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

grads = {"enc": jnp.array([3.0, 4.0]), "dec": jnp.array([0.0, 0.0, 1.0])}

clipped, norm = clip_by_global_norm(grads, max_norm=1.0)
print("global norm:", norm)                       # sqrt(25 + 1) = 5.099
print("global-clipped:", clipped)

# What per-tensor clipping would have done instead:
per_tensor = jax.tree.map(
    lambda g: g * jnp.minimum(1.0, 1.0 / (jnp.linalg.norm(g) + 1e-6)), grads
)
print("per-tensor    :", per_tensor)

flat = lambda t: jnp.concatenate([l.ravel() for l in jax.tree.leaves(t)])
cos = lambda a, b: jnp.dot(flat(a), flat(b)) / (
    jnp.linalg.norm(flat(a)) * jnp.linalg.norm(flat(b))
)
print("cos(g, global)     =", cos(grads, clipped))      # exactly 1.0
print("cos(g, per-tensor) =", cos(grads, per_tensor))   # < 1.0 -> direction moved

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gradient_clipping")